In [1]:
!rm -rf /content/PhaseForge
!git clone -b master https://github.com/Luke23-45/PhaseForge.git

Cloning into 'PhaseForge'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 145 (delta 28), reused 140 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 96.49 KiB | 844.00 KiB/s, done.
Resolving deltas: 100% (28/28), done.


In [2]:
%cd PhaseForge

/content/PhaseForge


In [3]:
%pip install -e ".[dev]"

Obtaining file:///content/PhaseForge
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached pathspec-1.1.1-py3-none-any.whl.metadata (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 131.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.8/531.8 kB 49.8 MB/s eta 0:00:00
Using cached pathspec-1.1.1-py3-none-any.whl (57 kB)
  Building editable for phaseforge (pyproject.toml) ... done
  Created wheel for phaseforge: filename=phaseforge-0.

In [14]:
import os
# Change directory to the repository
os.chdir('/content/PhaseForge')

# Stash local changes and pull
!git stash
!git pull

No local changes to save
Already up to date.


In [16]:
!rm -rf /content/PhaseForge/outputs


### 1. Data Generation & Pipeline Simulation
Verify the pipeline end-to-end locally using synthesized LIBERO HDF5 files.

In [5]:
!python scripts/simulate_pipeline.py


========== STEP 1: synthesize LIBERO-shaped HDF5 ==========
  wrote KITCHEN_SCENE1_open_drawer_demo.hdf5  (3 demos x 60 steps)
  wrote LIVING_ROOM_SCENE2_put_bowl_demo.hdf5  (3 demos x 60 steps)
  wrote STUDY_SCENE1_pick_book_demo.hdf5  (3 demos x 60 steps)
  -> /content/PhaseForge/_simulation/raw/libero/libero_90

========== STEP 2: build config ==========
  config built.

========== STEP 3: run FSM (download bypassed) ==========
  initial state: CHECK_PERSISTENT_CACHE
  config_hash: 301daa29da177b21
  [BYPASS] patched _validate_source to use synthetic HDF5
  run() returned without error.
    train: DataLoader
    val: DataLoader
    eval: None

========== STEP 4: validate output tensors ==========
    state: shape=(4, 23) dtype=torch.float32
    action: shape=(4, 7) dtype=torch.float32
    phase: shape=(4,) dtype=torch.int64
    task_id: shape=(4,) dtype=torch.int64

========== DONE ==========


### Download the dataset

In [6]:
!python scripts/download_and_extract.py --repo-id "hellxhell/phaseforge-cache"


Found 2 archives to download.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(

data/301daa29da177b21.tar.gz: downloading bytes:  83% 44.5k/53.7k [00:01<00:00, 52.1kB/s, 1.66kB/s  ]
data/301daa29da177b21.tar.gz: downloading bytes: 100% 53.7k/53.7k [00:01<00:00, 50.2kB/s, 5.23kB/s  ]
data/301daa29da177b21.tar.gz: reconstructing file: 100% 53.7k/53.7k [00:01<00:00, 50.2kB/s, 5.25kB/s  ]
Extracting 301daa29da177b21.tar.gz...
/content/PhaseForge/scripts/download_and_extract.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)
Cleaning up 301daa29da177b21.tar.gz to save space...
Successfully downloaded and extracted 301da

### 2. Main Experiment: Phase-Bootstrapped MoE

**Stage 1: Phase-Supervised Generalist Pretraining**

In [17]:
!phaseforge-train models=phaseforge train=stage1

[2026-07-17 06:27:32,751][phaseforge.cli][INFO] - Output directory: outputs/2026-07-17_06-27-32
[2026-07-17 06:27:32,751][phaseforge.cli][INFO] - Initializing Data Pipeline...
[2026-07-17 06:27:32,756][phaseforge.data.ingestion.state_machine][INFO] - Pipeline starting. Config hash: a4c74be17f117a4b
[2026-07-17 06:27:32,756][phaseforge.data.ingestion.state_machine][INFO] - CHECK_PERSISTENT_CACHE: looking for cached data…
[2026-07-17 06:27:32,756][phaseforge.data.ingestion.state_machine][INFO] - Cache hit. Loading from disk.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_ration

**Stage 2: MoE Bootstrapping**
*Auto-detects the latest checkpoint from `outputs/phaseforge/stage1/`.*
*Override explicitly with `train.stage1_ckpt_path=<path>` if needed.*


In [18]:
!phaseforge-train models=phaseforge train=stage2


### 3. Baselines

**BC Baseline (Standard Stage 1)**

In [19]:
!phaseforge-train models=baselines/bc train=stage1

[2026-07-17 06:46:30,182][phaseforge.cli][INFO] - Output directory: outputs/2026-07-17_06-46-30
[2026-07-17 06:46:30,182][phaseforge.cli][INFO] - Initializing Data Pipeline...
[2026-07-17 06:46:30,187][phaseforge.data.ingestion.state_machine][INFO] - Pipeline starting. Config hash: a4c74be17f117a4b
[2026-07-17 06:46:30,187][phaseforge.data.ingestion.state_machine][INFO] - CHECK_PERSISTENT_CACHE: looking for cached data…
[2026-07-17 06:46:30,187][phaseforge.data.ingestion.state_machine][INFO] - Cache hit. Loading from disk.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_ration

**Scratch MoE Baseline**

In [ ]:
!phaseforge-train models=baselines/scratch_moe train=stage2

**Warm-Start MoE Baseline**
*Auto-detects the latest BC checkpoint from `outputs/bc/stage1/`.*


In [ ]:
!phaseforge-train models=baselines/warmstart_moe train=stage2


**Oracle MoE Baseline**

In [ ]:
!phaseforge-train models=baselines/oracle_moe train=stage2

### 4. Evaluation
Evaluate trained models. Use the checkpoint discovery script to locate checkpoints.

**Find the latest checkpoint for a model:**
```
python scripts/find_checkpoint.py latest --model phaseforge --stage 2
```

**List all checkpoints across all models:**
```
python scripts/find_checkpoint.py list-all
```

*Replace the path below with your actual checkpoint path.*


In [ ]:
# Evaluate PhaseForge (use --stage 2 checkpoint)
!phaseforge-eval models=phaseforge train.stage1_ckpt_path=outputs/phaseforge/stage2/<ts>_<id>/checkpoints/checkpoint_best.pt

# Evaluate BC (use --stage 1 checkpoint)
# !phaseforge-eval models=baselines/bc train.stage1_ckpt_path=outputs/bc/stage1/<ts>_<id>/checkpoints/checkpoint_best.pt

# Evaluate Scratch MoE (use --stage 2 checkpoint)
# !phaseforge-eval models=baselines/scratch_moe train.stage1_ckpt_path=outputs/scratch_moe/stage2/<ts>_<id>/checkpoints/checkpoint_best.pt

# Evaluate Oracle MoE (use --stage 2 checkpoint)
# !phaseforge-eval models=baselines/oracle_moe train.stage1_ckpt_path=outputs/oracle_moe/stage2/<ts>_<id>/checkpoints/checkpoint_best.pt

# Evaluate WarmStart MoE (use --stage 2 checkpoint)
# !phaseforge-eval models=baselines/warmstart_moe train.stage1_ckpt_path=outputs/warmstart_moe/stage2/<ts>_<id>/checkpoints/checkpoint_best.pt


In [ ]:
import os
from google.colab import files

output_dir = '/content/PhaseForge/outputs'
zip_name = 'phaseforge_output.zip'

if os.path.exists(output_dir):
    !zip -r {zip_name} {output_dir}
    files.download(zip_name)
else:
    print(f'Error: The directory {output_dir} does not exist.')


Error: The directory /content/ghost-bank/output does not exist.
